In [ ]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd()
    while ROOT != ROOT.parent and not (ROOT / 'docs' / 'references' / 'WoodID_plan.md').exists():
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
print(f'[ROOT] {ROOT}')


# WoodID Phase 1 — Task B Original vs Strict-by-Origin Comparison

Notebook này tái tạo bảng/plot so sánh Task B main giữa split gốc và split strict theo origin. Nguồn strict là `docs/reports/task_b_wandb_all_runs_report-hotfix.md`; nguồn data gốc là CSV đã sinh từ report Phase 1 không strict.


In [ ]:
if Path('/content').exists():
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'matplotlib', 'seaborn'])


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUT = ROOT / 'docs' / 'reports' / 'phase_1_task_b_strict_comparison'
OUT.mkdir(parents=True, exist_ok=True)
MD = ROOT / 'docs' / 'reports' / 'task_b_wandb_all_runs_report-hotfix.md'
ORIG_DIR = ROOT / 'docs' / 'reports' / 'phase_1_run_report_not_strict'
BACKBONE_ORDER = ['convnext_tiny', 'swin_tiny', 'efficientnet_b3', 'mobilenetv3_large', 'resnet50']
DISPLAY = {'convnext_tiny':'ConvNeXt-Tiny','swin_tiny':'Swin-Tiny','efficientnet_b3':'EfficientNet-B3','mobilenetv3_large':'MobileNetV3-L','resnet50':'ResNet-50'}
sns.set_theme(style='whitegrid', context='notebook')


In [ ]:
def pct_to_float(value):
    s = str(value).strip()
    return None if s in {'', '-', 'nan'} else float(s.replace('%', '').replace(',', ''))

def parse_table(section_text):
    lines, started = [], False
    for line in section_text.splitlines():
        if line.startswith('|'):
            lines.append(line); started = True
        elif started:
            break
    header = [c.strip() for c in lines[0].strip('|').split('|')]
    rows = []
    for line in lines[2:]:
        cells = [c.strip() for c in line.strip('|').split('|')]
        if len(cells) == len(header):
            rows.append(dict(zip(header, cells)))
    return pd.DataFrame(rows)

def md_section(text, name, next_name):
    part = text.split(f'## {name}', 1)[1]
    return part.split(f'## {next_name}', 1)[0] if next_name else part

def clean_run_name(value):
    match = re.match(r'\[([^\]]+)\]', str(value))
    return match.group(1) if match else str(value)


In [ ]:
md_text = MD.read_text(encoding='utf-8')
strict_main = parse_table(md_section(md_text, 'M1_main / training', 'M1_main / test-eval extra'))
strict_main['run_name'] = strict_main['Run'].map(clean_run_name)
strict_main['backbone'] = strict_main['Backbone']
strict_main['seed'] = pd.to_numeric(strict_main['Seed'], errors='coerce')
strict_main['best_epoch'] = pd.to_numeric(strict_main['Best Epoch'].replace('-', pd.NA), errors='coerce')
strict_main['epochs_trained'] = pd.to_numeric(strict_main['Epochs'].replace('-', pd.NA), errors='coerce')
for src, dst in [('Val Acc (best/final)', 'best_val_acc'), ('Test Acc', 'test_accuracy'), ('Test F1', 'test_f1_macro'), ('Test Prec', 'test_precision_macro'), ('Test Rec', 'test_recall_macro')]:
    strict_main[dst] = strict_main[src].map(pct_to_float)
strict_main['display_backbone'] = strict_main['backbone'].map(DISPLAY)
strict_detail = strict_main[['run_name','backbone','display_backbone','seed','best_epoch','epochs_trained','best_val_acc','test_accuracy','test_f1_macro','test_precision_macro','test_recall_macro']]
strict_detail.sort_values(['backbone','seed'])


In [ ]:
strict_summary = strict_detail.groupby('backbone').agg(
    best_epoch_mean=('best_epoch','mean'), best_epoch_std=('best_epoch','std'),
    best_val_acc_mean=('best_val_acc','mean'), best_val_acc_std=('best_val_acc','std'),
    test_accuracy_mean=('test_accuracy','mean'), test_accuracy_std=('test_accuracy','std'),
    test_f1_macro_mean=('test_f1_macro','mean'), test_f1_macro_std=('test_f1_macro','std'),
    test_precision_macro_mean=('test_precision_macro','mean'), test_precision_macro_std=('test_precision_macro','std'),
    test_recall_macro_mean=('test_recall_macro','mean'), test_recall_macro_std=('test_recall_macro','std'),
    count=('test_f1_macro','count')).reset_index()
strict_summary['display_backbone'] = strict_summary['backbone'].map(DISPLAY)
strict_summary['order'] = strict_summary['backbone'].map({b:i for i,b in enumerate(BACKBONE_ORDER)})
strict_summary = strict_summary.sort_values('order').drop(columns='order')
strict_summary.round(2)


In [ ]:
orig = pd.read_csv(ORIG_DIR / 'phase1_summary_by_group_backbone_percent.csv')
orig = orig[orig['group'].eq('Task B main')].copy()
orig['display_backbone'] = orig['backbone'].map(DISPLAY)
orig['order'] = orig['backbone'].map({b:i for i,b in enumerate(BACKBONE_ORDER)})
orig = orig.sort_values('order').drop(columns='order')
comparison = orig[['backbone','display_backbone','test_accuracy_mean','test_f1_macro_mean','best_val_acc_mean','best_epoch_mean']].rename(columns={
    'test_accuracy_mean':'original_test_acc_mean','test_f1_macro_mean':'original_f1_mean','best_val_acc_mean':'original_val_acc_mean','best_epoch_mean':'original_best_epoch_mean'}).merge(
    strict_summary[['backbone','test_accuracy_mean','test_f1_macro_mean','best_val_acc_mean','best_epoch_mean']].rename(columns={
        'test_accuracy_mean':'strict_test_acc_mean','test_f1_macro_mean':'strict_f1_mean','best_val_acc_mean':'strict_val_acc_mean','best_epoch_mean':'strict_best_epoch_mean'}), on='backbone')
comparison['delta_test_acc_pp'] = comparison['strict_test_acc_mean'] - comparison['original_test_acc_mean']
comparison['delta_f1_pp'] = comparison['strict_f1_mean'] - comparison['original_f1_mean']
comparison['delta_val_acc_pp'] = comparison['strict_val_acc_mean'] - comparison['original_val_acc_mean']
comparison['delta_best_epoch'] = comparison['strict_best_epoch_mean'] - comparison['original_best_epoch_mean']
comparison.round(2)


In [ ]:
strict_detail.to_csv(OUT / 'strict_task_b_main_detail.csv', index=False)
strict_summary.to_csv(OUT / 'strict_task_b_main_summary.csv', index=False)
comparison.to_csv(OUT / 'task_b_original_vs_strict_comparison.csv', index=False)
print(f'Wrote CSVs to {OUT}')


In [ ]:
plot_df = comparison[['display_backbone','original_test_acc_mean','strict_test_acc_mean','original_f1_mean','strict_f1_mean']].melt(id_vars='display_backbone', var_name='metric_source', value_name='score')
plot_df['Metric'] = plot_df['metric_source'].map({'original_test_acc_mean':'Test Acc','strict_test_acc_mean':'Test Acc','original_f1_mean':'F1-macro','strict_f1_mean':'F1-macro'})
plot_df['Split'] = plot_df['metric_source'].map(lambda x: 'Original split' if x.startswith('original') else 'Strict-by-origin split')
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, metric in zip(axes, ['Test Acc', 'F1-macro']):
    sub = plot_df[plot_df['Metric'].eq(metric)]
    sns.barplot(data=sub, x='display_backbone', y='score', hue='Split', ax=ax, palette=['#4C78A8','#F58518'])
    ax.set_title(f'Task B Main: {metric}')
    ax.set_xlabel('')
    ax.set_ylabel('Score (%)' if ax is axes[0] else '')
    ax.set_ylim(80, 101)
    ax.tick_params(axis='x', rotation=25)
    ax.legend(loc='lower left', fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'task_b_original_vs_strict_acc_f1.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
delta = comparison.sort_values('delta_f1_pp')
colors = ['#D55E00' if v < 0 else '#009E73' for v in delta['delta_f1_pp']]
ax.bar(delta['display_backbone'], delta['delta_f1_pp'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Task B Main: F1 change after strict-by-origin split')
ax.set_ylabel('Strict - Original F1 (percentage points)')
ax.tick_params(axis='x', rotation=25)
for i, v in enumerate(delta['delta_f1_pp']):
    ax.text(i, v - 0.35 if v < 0 else v + 0.1, f'{v:+.2f}', ha='center', va='top' if v < 0 else 'bottom', fontsize=9)
fig.tight_layout()
fig.savefig(OUT / 'task_b_strict_delta_f1.png', dpi=180, bbox_inches='tight')
plt.show()


## Interpretation

Strict-by-origin split làm Task B main khó hơn rõ rệt cho các backbone kém robust hơn. ConvNeXt-Tiny và Swin-Tiny vẫn giữ F1 rất cao, trong khi ResNet-50, EfficientNet-B3 và MobileNetV3-Large giảm mạnh hơn. Điều này cho thấy split gốc nhiều khả năng có tương quan theo origin giúp mô hình tổng quát hóa dễ hơn so với điều kiện origin-disjoint.
